In [1]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("silver_brasileirao") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO configurados!")

Spark version: 3.5.0
Delta + MinIO configurados!


In [2]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
LEAGUE = "brasileirao-serie-a"
SEASON = "2026"
ENVIRONMENT = "dev"
REPROC_MODE = False

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"
SILVER_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/silver/{LEAGUE}"

DEDUP_KEYS = {
    "standing": {
        "pk": ["season", "record.teamId"],
        "order_col": "ingested_at"
    },
    "rounds": {
        "pk": ["record.id"],
        "order_col": "ingested_at"
    },
    "events": {
        "pk": ["record.events.id"],
        "order_col": "ingested_at"
    },
    "players": {
        "pk": ["record.players.id", "record.teams.teamId", "season"],
        "order_col": "ingested_at"
    },
    "team_stats": {
        "pk": ["record.team.teamId", "season"],
        "order_col": "ingested_at"
    }
}

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Raw:    {RAW_PATH}")
print(f"Silver: {SILVER_PATH}")

League: brasileirao-serie-a | Season: 2026 | Env: dev | Reproc: False
Raw:    s3a://eng-prd-data-lake/dev/raw/brasileirao-serie-a
Silver: s3a://eng-prd-data-lake/dev/silver/brasileirao-serie-a


In [3]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

def raw_to_silver(data_type):
    raw = f"{RAW_PATH}/{data_type}"
    silver = f"{SILVER_PATH}/{data_type}"
    config = DEDUP_KEYS[data_type]
    pks = config["pk"]
    order_col = config["order_col"]

    print(f"Lendo {data_type} da Raw...")
    df = spark.read.format("delta").load(raw)

    # Deduplicação — mantém o registro mais recente por chave
    window = Window.partitionBy([col(pk) for pk in pks]) \
                   .orderBy(col(order_col).desc())

    df_dedup = df.withColumn("_rank", row_number().over(window)) \
                 .filter(col("_rank") == 1) \
                 .drop("_rank")

    print(f"Registros após dedup: {df_dedup.count()}")

    df_dedup.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("season") \
        .save(silver)

    print(f"[OK] {data_type} salvo na Silver!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats"]:
    raw_to_silver(data_type)

print("Camada Silver finalizada!")

Lendo standing da Raw...
Registros após dedup: 20
[OK] standing salvo na Silver!

Lendo rounds da Raw...
Registros após dedup: 7
[OK] rounds salvo na Silver!

Lendo events da Raw...
Registros após dedup: 10
[OK] events salvo na Silver!

Lendo players da Raw...
Registros após dedup: 327
[OK] players salvo na Silver!

Lendo team_stats da Raw...
Registros após dedup: 1
[OK] team_stats salvo na Silver!

Camada Silver finalizada!


In [7]:
from pyspark.sql.functions import explode, col, lit

df = spark.read.format("delta").load(f"{SILVER_PATH}/team_stats")

STAT_KEYS = [
    "accurateCross", "bigChanceCreated", "bigChanceMissed", "bigChanceScored",
    "expectedGoals", "shotsOnGoal", "shotsOffGoal", "totalShotsInsideBox", "totalShotsOutsideBox",
    "totalClearance", "dispossessed", "errorsLeadToGoal", "errorsLeadToShot",
    "fouls", "goalkeeperSaves", "interceptionWon", "totalTackle",
    "freeKicks", "goalKicks", "throwIns",
    "ballPossession", "offsides", "passes", "touchesInOppBox",
    "redCards", "yellowCards"
]

EVENT_TYPES = ["all", "home", "away"]

dfs = []

for event_type in EVENT_TYPES:
    for key in STAT_KEYS:
        df_temp = df.select(
            col("record.team.teamId").alias("team_id"),
            col("record.team.teamName").alias("team_name"),
            col("record.team.teamSlug").alias("team_slug"),
            lit(event_type).alias("event_type"),
            explode(col(f"record.stats.{event_type}.{key}")).alias("stat")
        ).select(
            "team_id", "team_name", "team_slug", "event_type",
            col("stat.statisticKey"),
            col("stat.event_id"),
            col("stat.home_team_id"),
            col("stat.home_team_name"),
            col("stat.away_team_id"),
            col("stat.away_team_name"),
            col("stat.home_score"),
            col("stat.away_score"),
            col("stat.home_value").cast("double").alias("home_value"),
            col("stat.away_value").cast("double").alias("away_value"),
            col("stat.time_start_timestamp")
        )
        dfs.append(df_temp)

from functools import reduce
from pyspark.sql import DataFrame

df_final = reduce(DataFrame.unionByName, dfs)
print(f"Total de registros: {df_final.count()}")
df_final.show(10, truncate=False)

Total de registros: 3712
+-------+-----------+-----------+----------+-------------+--------+------------+--------------+------------+-------------------+----------+----------+----------+----------+--------------------+
|team_id|team_name  |team_slug  |event_type|statisticKey |event_id|home_team_id|home_team_name|away_team_id|away_team_name     |home_score|away_score|home_value|away_value|time_start_timestamp|
+-------+-----------+-----------+----------+-------------+--------+------------+--------------+------------+-------------------+----------+----------+----------+----------+--------------------+
|1957   |Corinthians|corinthians|all       |accurateCross|15237919|1954        |Cruzeiro      |1957        |Corinthians        |1         |1         |1.0       |5.0       |1772060400          |
|1957   |Corinthians|corinthians|all       |accurateCross|15369877|1967        |Athletico     |1957        |Corinthians        |0         |1         |3.0       |0.0       |1771540200          |
|1957